In [2]:
import numpy as np

In [ ]:
class Neural:
  def __init__(self,input_size=784,hidden_layers=[128,128],output_size=10):
    self.input_size = input_size
    self.hidden_layers = hidden_layers
    self.output_size = output_size
    self.weights = []
    self.biases = []
    self.layers = [] # Post-ReLU values
    self.layers_z = [] #Pre-ReLU values

    #Input for hidden layers
    self.weights.append(0.1 * np.random.randn(input_size,hidden_layers[0]))
    self.biases.append(np.zeros((1,hidden_layers[0])))

    #Hidden Layers Network
    for i in range(len(hidden_layers)-1):
      self.weights.append(0.1 * np.random.randn(hidden_layers[i],hidden_layers[i+1]))
      self.biases.append(np.zeros((1,hidden_layers[i+1])))

    #Hidden layers to Output
    self.weights.append(0.1 * np.random.randn(hidden_layers[-1],output_size))
    self.biases.append(np.zeros((1,output_size)))

  def forward(self,inputs):
    self.layers = [inputs]
    self.layers_z = []
    #Dot product of inputs and weights
    for i in range(len(self.weights)):
      if i == (len(self.weights)-1):
        self.layers_z.append(np.dot(self.layers[-1],self.weights[i])+self.biases[i])
      else: 
        output = np.dot(self.layers[-1],self.weights[i])+self.biases[i]
        self.layers_z.append(output)
        output = self.relu(output)
        self.layers.append(output)
    self.layers.append(self.softmax_act(self.layers_z[-1]))
    return self.layers[-1]

  def relu(self,inputs):
    return np.where(inputs < 0, 0, inputs)

  def softmax_act(self,inputs):
    exp_values = np.exp(inputs)
    return exp_values / np.sum(exp_values,axis=1,keepdims=True)

  def loss(self,input,target,epsilon=1e-15):
    # Clip predictions to avoid numerical instability (log(0))
    y_pred = np.clip(input, epsilon, 1 - epsilon)
    
    # Calculate loss for each sample and then the average over the batch
    loss = -np.mean(np.sum(target * np.log(y_pred), axis=-1))
    return loss
  
  def backpass(self,inputs,target,batch_size):
    dOutput = np.subtract(self.layers[-1],target)
    dgradient = dOutput/batch_size
    d_weights_list = []
    d_biases_list = []
    for i in range(len(self.weights)-1,-1,-1):
      dweights = np.dot(self.layers[i].T, dgradient)
      dbiases = np.sum(dgradient,axis=0)
      dgradient = np.dot(dgradient, self.weights[i].T)
      if i != (len(self.weights)-1) and i!=0:
        dgradient = np.multiply(dgradient,self.layers_z[i-1]>0)
      d_weights_list.append(dweights)
      d_biases_list.append(dbiases)
    d_weights_list = d_weights_list[::-1]
    d_biases_list = d_biases_list[::-1]
    return d_weights_list, d_biases_list

  def update(self, d_weights_list, d_biases_list, learning_rate=0.01):
    for i in range(len(self.weights)):
        self.weights[i] -= learning_rate * d_weights_list[i]
        self.biases[i]  -= learning_rate * d_biases_list[i]

  def train(self,inputs,target,batch_size,epochs,learning_rate):
    for i in range(epochs):
      print(f"Epoch {i}")
      f_pass = self.forward(inputs)
      loss = self.loss(f_pass,target)
      print(f"Loss {loss}")
      d_w , d_l = self.backpass(inputs,target,batch_size)
      self.update(d_w,d_l,learning_rate)


In [4]:
img_data = np.load("Datasets/train_images.npy")
img_labels = np.load("Datasets/train_labels.npy")

In [5]:
img_data = np.reshape(img_data,(60000,784))
img_labels = np.reshape(img_labels,(60000,1))

In [6]:
img_norm = img_data/255

In [7]:
def true_vals(array):
  array = array.flatten()
  true_values = np.zeros((array.size,array.max()+1))
  true_values[np.arange(array.size),array] = 1
  return true_values

In [8]:
labels = true_vals(img_labels)

In [ ]:
neural_net = Neural()
neural_net.train(img_norm, labels, batch_size=60000, epochs=50, learning_rate=0.1)

Epoch 0
Loss 2.3473289198303524
Epoch 1
Loss 2.3207997194996985
Epoch 2
Loss 2.297575575599489
Epoch 3
Loss 2.2760896309245493
Epoch 4
Loss 2.2554812858254216
Epoch 5
Loss 2.2351819068335153
Epoch 6
Loss 2.214791777044287
Epoch 7
Loss 2.1940117485691317
Epoch 8
Loss 2.1725811676420905
Epoch 9
Loss 2.150311971647245
Epoch 10
Loss 2.1270606687450404
Epoch 11
Loss 2.102715062426386
Epoch 12
Loss 2.077200133355671
Epoch 13
Loss 2.050467722584435
Epoch 14
Loss 2.0224965703837214
Epoch 15
Loss 1.993289242002097
Epoch 16
Loss 1.9628679179047426
Epoch 17
Loss 1.9312670457369239
Epoch 18
Loss 1.8985585787379908
Epoch 19
Loss 1.8648511337526505
Epoch 20
Loss 1.830264954485562
Epoch 21
Loss 1.7949107360825836
Epoch 22
Loss 1.7589238600778025
Epoch 23
Loss 1.722464467040624
Epoch 24
Loss 1.6856770335003008
Epoch 25
Loss 1.6487293217096852
Epoch 26
Loss 1.6117473772690631
Epoch 27
Loss 1.5748407277340708
Epoch 28
Loss 1.5381113917498102
Epoch 29
Loss 1.5016604109731087
Epoch 30
Loss 1.4655820611115